In [1]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

2026-06-28 10:14:26.061639: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782641666.253849      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782641666.300851      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782641666.694813      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782641666.694852      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782641666.694854      58 computation_placer.cc:177] computation placer alr

In [2]:
# Importing ResNet50

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import GlobalMaxPooling2D, Input
from tensorflow.keras.models import Model

In [3]:
CSV_PATH = "/kaggle/input/notebooks/sumedhabhadauria/01-dataset-preparation/clean_metadata.csv"
IMAGE_FOLDER = Path("/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/images")

In [4]:
# Loading Metadata

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(44419, 11)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans,/kaggle/input/datasets/paramaggarwal/fashion-p...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch,/kaggle/input/datasets/paramaggarwal/fashion-p...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants,/kaggle/input/datasets/paramaggarwal/fashion-p...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...


In [5]:
# Loading ResNet50

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False
inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
outputs = GlobalMaxPooling2D()(x)
feature_extractor = Model(inputs, outputs)

I0000 00:00:1782641898.416255      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782641898.422209      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [6]:
# Verifying Model

feature_extractor.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 2048)           │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [7]:
# Feature Extarction Function

def extract_feature(img_path):

    img = image.load_img(
        img_path,
        target_size=(224,224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    feature = feature_extractor.predict(
        img,
        verbose=0
    ).flatten()
    feature = feature / np.linalg.norm(feature)
    return feature

In [8]:
sample_image = IMAGE_FOLDER / f"{df.iloc[0]['id']}.jpg"
feature = extract_feature(sample_image)
print(feature.shape)

I0000 00:00:1782641999.231619     185 service.cc:152] XLA service 0x7dca0c0034c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782641999.231660     185 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782641999.231664     185 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782642000.056964     185 cuda_dnn.cc:529] Loaded cuDNN version 91002


(2048,)


I0000 00:00:1782642002.667861     185 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [11]:
# Embeddings Generation

embeddings = []
image_ids = []
skipped_images = []

for product_id in tqdm(df["id"], desc="Extracting Features"):
    img_path = IMAGE_FOLDER / f"{product_id}.jpg"
    if not img_path.exists():
        skipped_images.append(product_id)
        continue
    try:
        embedding = extract_feature(img_path)
        embeddings.append(embedding)
        image_ids.append(product_id)
    except Exception:
        skipped_images.append(product_id)
embeddings = np.array(embeddings)
print("\nFeature Extraction Completed\n")
print(f"Embeddings Shape : {embeddings.shape}")
print(f"Processed Images : {len(image_ids)}")
print(f"Skipped Images   : {len(skipped_images)}")

Extracting Features:   0%|          | 0/44419 [00:00<?, ?it/s]


Feature Extraction Completed

Embeddings Shape : (44419, 2048)
Processed Images : 44419
Skipped Images   : 0


In [12]:
# Saving Embeddings

np.save("/kaggle/working/embeddings.npy",embeddings)

with open("/kaggle/working/image_ids.pkl", "wb") as f:
    pickle.dump(image_ids, f)

feature_extractor.save("/kaggle/working/feature_extractor.keras")

In [13]:
print(f"Embedding Dimension : {embeddings.shape[1]}")
print(f"Total Embeddings    : {len(embeddings)}")
print(f"Total Image IDs     : {len(image_ids)}")
print(f"Skipped Images      : {len(skipped_images)}")

Embedding Dimension : 2048
Total Embeddings    : 44419
Total Image IDs     : 44419
Skipped Images      : 0
